# Notebook 50: Hourly Timeframe Strategy Testing

Test STRAT-002 at different timeframes to find optimal resolution.

**Approaches tested:**
1. Daily signals + Daily trail (baseline)
2. Daily signals + Hourly trail (hybrid)
3. 4-Hourly signals + 4H trail
4. 4-Hourly MA-smoothed signals + 4H trail

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import vectorbt as vbt
from pathlib import Path

# Paths
DATA_DIR = Path("../data")
DAILY_DIR = DATA_DIR / "daily"
HOURLY_DIR = DATA_DIR / "hourly"

## 1. Load Data

In [ ]:
# Load hourly data
def load_hourly_full():
    price = pd.read_parquet(HOURLY_DIR / "price.parquet")
    sopr = pd.read_parquet(HOURLY_DIR / "sopr.parquet")
    sopr_sth = pd.read_parquet(HOURLY_DIR / "sopr_sth.parquet")
    sopr_lth = pd.read_parquet(HOURLY_DIR / "sopr_lth.parquet")
    realized_loss = pd.read_parquet(HOURLY_DIR / "realized_loss.parquet")
    
    df = price.rename(columns={"value": "price"}).set_index("time")
    df["sopr"] = sopr.set_index("time")["value"]
    df["sopr_sth"] = sopr_sth.set_index("time")["value"]
    df["sopr_lth"] = sopr_lth.set_index("time")["value"]
    df["realized_loss"] = realized_loss.set_index("time")["value"]
    
    return df

# Load daily data
def load_daily():
    price = pd.read_parquet(DAILY_DIR / "price.parquet")
    sopr = pd.read_parquet(DAILY_DIR / "sopr.parquet")
    sopr_sth = pd.read_parquet(DAILY_DIR / "sopr_sth.parquet")
    sopr_lth = pd.read_parquet(DAILY_DIR / "sopr_lth.parquet")
    realized_loss = pd.read_parquet(DAILY_DIR / "realized_loss.parquet")
    
    df = price.rename(columns={"value": "price"}).set_index("time")
    df["sopr"] = sopr.set_index("time")["value"]
    df["sopr_sth"] = sopr_sth.set_index("time")["value"]
    df["sopr_lth"] = sopr_lth.set_index("time")["value"]
    df["realized_loss"] = realized_loss.set_index("time")["value"]
    
    # Z-score
    df["rl_mean"] = df["realized_loss"].rolling(window=365, min_periods=180).mean()
    df["rl_std"] = df["realized_loss"].rolling(window=365, min_periods=180).std()
    df["rl_zscore"] = (df["realized_loss"] - df["rl_mean"]) / df["rl_std"]
    
    return df

df_hourly = load_hourly_full()
df_daily = load_daily()

print(f"Hourly data: {len(df_hourly):,} rows")
print(f"Daily data: {len(df_daily):,} rows")

In [ ]:
# Create 4-hourly data by resampling hourly
def resample_to_4h(df_hourly):
    """Resample hourly data to 4-hourly."""
    df_4h = pd.DataFrame()
    
    # Price: use close of 4h period
    df_4h["price"] = df_hourly["price"].resample("4h").last()
    
    # On-chain metrics: use mean of 4h period (aggregated behavior)
    df_4h["sopr"] = df_hourly["sopr"].resample("4h").mean()
    df_4h["sopr_sth"] = df_hourly["sopr_sth"].resample("4h").mean()
    df_4h["sopr_lth"] = df_hourly["sopr_lth"].resample("4h").mean()
    df_4h["realized_loss"] = df_hourly["realized_loss"].resample("4h").sum()  # Sum for volume-like metrics
    
    return df_4h.dropna()

df_4h = resample_to_4h(df_hourly)
print(f"4-Hourly data: {len(df_4h):,} rows")
print(f"Date range: {df_4h.index.min()} to {df_4h.index.max()}")

In [ ]:
# Filter to backtest period
START_DATE = "2019-01-01"

df_daily = df_daily[df_daily.index >= START_DATE].dropna()
df_hourly = df_hourly[df_hourly.index >= START_DATE].ffill(limit=4).dropna()
df_4h = df_4h[df_4h.index >= START_DATE].ffill(limit=2).dropna()

# Calculate z-scores
# Daily: 365-day window
# 4H: 365*6 = 2190 periods (6 bars per day)
# Hourly: 365*24 = 8760 periods

WINDOW_4H = 365 * 6  # 1 year in 4h bars
df_4h["rl_mean"] = df_4h["realized_loss"].rolling(window=WINDOW_4H, min_periods=WINDOW_4H//2).mean()
df_4h["rl_std"] = df_4h["realized_loss"].rolling(window=WINDOW_4H, min_periods=WINDOW_4H//2).std()
df_4h["rl_zscore"] = (df_4h["realized_loss"] - df_4h["rl_mean"]) / df_4h["rl_std"]

WINDOW_H = 365 * 24
df_hourly["rl_mean"] = df_hourly["realized_loss"].rolling(window=WINDOW_H, min_periods=WINDOW_H//2).mean()
df_hourly["rl_std"] = df_hourly["realized_loss"].rolling(window=WINDOW_H, min_periods=WINDOW_H//2).std()
df_hourly["rl_zscore"] = (df_hourly["realized_loss"] - df_hourly["rl_mean"]) / df_hourly["rl_std"]

print(f"Daily rows: {len(df_daily):,}")
print(f"4H rows: {len(df_4h):,}")
print(f"Hourly rows: {len(df_hourly):,}")

## 2. Helper Functions

In [ ]:
def get_metrics(pf, years):
    """Extract key metrics from portfolio."""
    trades = pf.trades.records_readable
    if len(trades) == 0:
        return None
    
    total_return = pf.total_return() * 100
    
    return {
        "return": total_return,
        "cagr": ((1 + total_return/100) ** (1/years) - 1) * 100,
        "sharpe": pf.sharpe_ratio(),
        "max_dd": pf.max_drawdown() * 100,
        "trades": len(trades),
        "win_rate": (trades["PnL"] > 0).mean() * 100,
        "profit_factor": abs(trades[trades["PnL"] > 0]["PnL"].sum() / trades[trades["PnL"] < 0]["PnL"].sum()) if (trades["PnL"] < 0).any() else np.inf
    }

years = (df_daily.index.max() - df_daily.index.min()).days / 365.25
bh_return = (df_daily["price"].iloc[-1] / df_daily["price"].iloc[0] - 1) * 100
print(f"Backtest: {years:.2f} years | Buy & Hold: {bh_return:+,.0f}%")

## 3. Baseline: Daily Signals + Daily Trail

In [ ]:
# Daily entry
daily_cond = (df_daily["sopr"] < 1) & (df_daily["sopr_sth"] < 1) & (df_daily["rl_zscore"] > 0.5)
daily_entry = daily_cond & ~daily_cond.shift(1).fillna(False)

pf_daily = vbt.Portfolio.from_signals(
    close=df_daily["price"],
    entries=daily_entry,
    exits=None,
    sl_stop=0.30,
    sl_trail=True,
    freq="1d",
    init_cash=10000,
    fees=0.001
)

m_daily = get_metrics(pf_daily, years)
print(f"Daily baseline: {m_daily['return']:+,.0f}% | {m_daily['trades']} trades | {m_daily['win_rate']:.0f}% win")

## 4. Hybrid: Daily Signals + Hourly Trail

In [ ]:
# Map daily entries to hourly
daily_entry_dates = df_daily.index[daily_entry].tolist()
hourly_entry_hybrid = pd.Series(False, index=df_hourly.index)

for daily_date in daily_entry_dates:
    hourly_matches = df_hourly.index[df_hourly.index >= daily_date]
    if len(hourly_matches) > 0:
        hourly_entry_hybrid.loc[hourly_matches[0]] = True

pf_hybrid = vbt.Portfolio.from_signals(
    close=df_hourly["price"],
    entries=hourly_entry_hybrid,
    exits=None,
    sl_stop=0.30,
    sl_trail=True,
    freq="1h",
    init_cash=10000,
    fees=0.001
)

m_hybrid = get_metrics(pf_hybrid, years)
print(f"Hybrid (daily sig + hourly trail): {m_hybrid['return']:+,.0f}% | {m_hybrid['trades']} trades | {m_hybrid['win_rate']:.0f}% win")

## 5. 4-HOURLY: Raw Signals + 4H Trail

Test raw 4H on-chain signals without smoothing.

In [ ]:
# Raw 4H signals
cond_4h_raw = (df_4h["sopr"] < 1) & (df_4h["sopr_sth"] < 1) & (df_4h["rl_zscore"] > 0.5)
entry_4h_raw = cond_4h_raw & ~cond_4h_raw.shift(1).fillna(False)

print(f"Raw 4H entries: {entry_4h_raw.sum()}")

if entry_4h_raw.sum() > 0:
    pf_4h_raw = vbt.Portfolio.from_signals(
        close=df_4h["price"],
        entries=entry_4h_raw,
        exits=None,
        sl_stop=0.30,
        sl_trail=True,
        freq="4h",
        init_cash=10000,
        fees=0.001
    )
    m_4h_raw = get_metrics(pf_4h_raw, years)
    print(f"4H raw signals: {m_4h_raw['return']:+,.0f}% | {m_4h_raw['trades']} trades | {m_4h_raw['win_rate']:.0f}% win")
else:
    m_4h_raw = None
    print("No entries with raw 4H signals")

## 6. 4-HOURLY: MA-Smoothed Signals

Apply moving average smoothing to 4H on-chain data.

| MA Periods | 4H Bars | Equivalent |
|------------|---------|------------|
| 6 | 6 | 1 day |
| 12 | 12 | 2 days |
| 18 | 18 | 3 days |
| 42 | 42 | 1 week |

In [ ]:
# Test different MA periods on 4H data
ma_periods_4h = [3, 6, 12, 18, 24, 42]  # in 4H bars

print("\n" + "="*120)
print("4-HOURLY MA-SMOOTHED SIGNALS")
print("="*120)
print(f"\n{'MA (bars)':<12} {'~Days':<8} {'Return':>12} {'CAGR':>10} {'Sharpe':>10} {'MaxDD':>10} {'Trades':>8} {'WinRate':>10}")
print("-"*120)

ma_results_4h = {}

for ma in ma_periods_4h:
    # Apply MA smoothing
    sopr_ma = df_4h["sopr"].rolling(window=ma, min_periods=ma//2).mean()
    sopr_sth_ma = df_4h["sopr_sth"].rolling(window=ma, min_periods=ma//2).mean()
    rl_zscore_ma = df_4h["rl_zscore"].rolling(window=ma, min_periods=ma//2).mean()
    
    # Entry condition
    ma_cond = (sopr_ma < 1) & (sopr_sth_ma < 1) & (rl_zscore_ma > 0.5)
    ma_entry = ma_cond & ~ma_cond.shift(1).fillna(False)
    
    if ma_entry.sum() == 0:
        print(f"{ma:<12} {ma*4/24:.1f}d{'':<5} {'NO ENTRIES':>12}")
        continue
    
    pf = vbt.Portfolio.from_signals(
        close=df_4h["price"],
        entries=ma_entry,
        exits=None,
        sl_stop=0.30,
        sl_trail=True,
        freq="4h",
        init_cash=10000,
        fees=0.001
    )
    
    m = get_metrics(pf, years)
    if m:
        ma_results_4h[ma] = {"metrics": m, "pf": pf, "entry": ma_entry}
        days_equiv = ma * 4 / 24
        print(f"{ma:<12} {days_equiv:.1f}d{'':<5} {m['return']:>+11,.0f}% {m['cagr']:>+9.1f}% {m['sharpe']:>10.2f} {m['max_dd']:>9.1f}% {m['trades']:>8} {m['win_rate']:>9.0f}%")

print("-"*120)
print(f"{'Daily base':<12} {'':<8} {m_daily['return']:>+11,.0f}% {m_daily['cagr']:>+9.1f}% {m_daily['sharpe']:>10.2f} {m_daily['max_dd']:>9.1f}% {m_daily['trades']:>8} {m_daily['win_rate']:>9.0f}%")
print(f"{'Buy & Hold':<12} {'':<8} {bh_return:>+11,.0f}%")

## 7. EMA vs SMA on 4H

In [ ]:
# Test EMA vs SMA on 4H with best period from above
if ma_results_4h:
    best_ma_period = max(ma_results_4h.keys(), key=lambda x: ma_results_4h[x]['metrics']['return'])
else:
    best_ma_period = 12  # default

print(f"\nTesting EMA vs SMA with period={best_ma_period} (4H bars)")
print("="*80)

for ma_type in ["SMA", "EMA"]:
    if ma_type == "SMA":
        sopr_smooth = df_4h["sopr"].rolling(window=best_ma_period).mean()
        sopr_sth_smooth = df_4h["sopr_sth"].rolling(window=best_ma_period).mean()
        rl_zscore_smooth = df_4h["rl_zscore"].rolling(window=best_ma_period).mean()
    else:
        sopr_smooth = df_4h["sopr"].ewm(span=best_ma_period, adjust=False).mean()
        sopr_sth_smooth = df_4h["sopr_sth"].ewm(span=best_ma_period, adjust=False).mean()
        rl_zscore_smooth = df_4h["rl_zscore"].ewm(span=best_ma_period, adjust=False).mean()
    
    cond = (sopr_smooth < 1) & (sopr_sth_smooth < 1) & (rl_zscore_smooth > 0.5)
    entry = cond & ~cond.shift(1).fillna(False)
    
    if entry.sum() == 0:
        print(f"{ma_type}: NO ENTRIES")
        continue
    
    pf = vbt.Portfolio.from_signals(
        close=df_4h["price"],
        entries=entry,
        exits=None,
        sl_stop=0.30,
        sl_trail=True,
        freq="4h",
        init_cash=10000,
        fees=0.001
    )
    
    m = get_metrics(pf, years)
    if m:
        print(f"{ma_type}: {m['return']:+,.0f}% | Sharpe {m['sharpe']:.2f} | {m['trades']} trades | {m['win_rate']:.0f}% win")

## 8. Hybrid: Daily Signals + 4H Trail

In [ ]:
# Map daily entries to 4H
entry_4h_hybrid = pd.Series(False, index=df_4h.index)

for daily_date in daily_entry_dates:
    matches = df_4h.index[df_4h.index >= daily_date]
    if len(matches) > 0:
        entry_4h_hybrid.loc[matches[0]] = True

pf_4h_hybrid = vbt.Portfolio.from_signals(
    close=df_4h["price"],
    entries=entry_4h_hybrid,
    exits=None,
    sl_stop=0.30,
    sl_trail=True,
    freq="4h",
    init_cash=10000,
    fees=0.001
)

m_4h_hybrid = get_metrics(pf_4h_hybrid, years)
print(f"Hybrid (daily sig + 4H trail): {m_4h_hybrid['return']:+,.0f}% | {m_4h_hybrid['trades']} trades | {m_4h_hybrid['win_rate']:.0f}% win")

## 9. Final Comparison

In [ ]:
print("\n" + "="*120)
print("FINAL COMPARISON: ALL APPROACHES")
print("="*120)

all_results = {
    "Daily sig + Daily trail": m_daily,
    "Daily sig + Hourly trail": m_hybrid,
    "Daily sig + 4H trail": m_4h_hybrid,
}

if m_4h_raw:
    all_results["4H raw signals"] = m_4h_raw

# Add best MA-smoothed 4H
if ma_results_4h:
    best_ma = max(ma_results_4h.keys(), key=lambda x: ma_results_4h[x]['metrics']['return'])
    all_results[f"4H MA-{best_ma} smoothed"] = ma_results_4h[best_ma]['metrics']

print(f"\n{'Approach':<30} {'Return':>12} {'CAGR':>10} {'Sharpe':>10} {'MaxDD':>10} {'Trades':>8} {'WinRate':>10}")
print("-"*120)

for name, m in all_results.items():
    if m:
        print(f"{name:<30} {m['return']:>+11,.0f}% {m['cagr']:>+9.1f}% {m['sharpe']:>10.2f} {m['max_dd']:>9.1f}% {m['trades']:>8} {m['win_rate']:>9.0f}%")

print("-"*120)
print(f"{'Buy & Hold':<30} {bh_return:>+11,.0f}%")

# Winner
best_name = max(all_results.keys(), key=lambda x: all_results[x]['return'] if all_results[x] else 0)
print(f"\n🏆 WINNER: {best_name} with {all_results[best_name]['return']:+,.0f}% return")

In [ ]:
# Equity curves
fig, ax = plt.subplots(figsize=(14, 6))

pf_daily.value().plot(ax=ax, label=f"Daily ({m_daily['return']:+,.0f}%)", linewidth=2)
pf_4h_hybrid.value().resample('D').last().plot(ax=ax, label=f"Daily sig + 4H trail ({m_4h_hybrid['return']:+,.0f}%)", linewidth=2)

if ma_results_4h:
    best_pf = ma_results_4h[best_ma]['pf']
    best_pf.value().resample('D').last().plot(ax=ax, label=f"4H MA-{best_ma} ({ma_results_4h[best_ma]['metrics']['return']:+,.0f}%)", linewidth=2)

ax.set_title("Strategy Comparison")
ax.set_ylabel("Portfolio Value ($)")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_yscale('log')
plt.tight_layout()
plt.show()

## 10. Conclusions

In [ ]:
print("\n" + "="*80)
print("CONCLUSIONS")
print("="*80)

print("""
1. SIGNAL GENERATION:
   - Daily on-chain signals are most reliable (filtered noise)
   - 4H signals with MA smoothing may offer faster entries
   - Raw hourly/4H signals are too noisy

2. TRAIL EXECUTION:
   - Higher frequency trail (4H/hourly) can capture better exit prices
   - Trade-off: more frequent checks vs daily simplicity

3. RECOMMENDATION:
""")

# Compare daily vs 4H trail
if m_daily and m_4h_hybrid:
    diff = m_4h_hybrid['return'] - m_daily['return']
    if diff > 100:  # >100% improvement
        print(f"   → USE 4H TRAIL: +{diff:,.0f}% improvement over daily")
    elif diff > 0:
        print(f"   → 4H trail slightly better (+{diff:,.0f}%), but daily is simpler")
    else:
        print(f"   → STICK WITH DAILY: 4H trail cost {abs(diff):,.0f}%")